## 1. Notebook Setup

This section imports the required libraries, sets the project paths, and prepares the modelling tools needed for supervised vulnerability classification.

The notebook starts from the vulnerability-ready dataset created in Notebook 05.

In [5]:
# ==================================================
# Imports
# ==================================================

from pathlib import Path
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# ==================================================
# Machine Learning Tools
# ==================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ==================================================
# Project Paths
# ==================================================

PROJECT_ROOT = Path("..")

INTERIM_DATA = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "county_year_features"
)

RESULTS_TABLES = (
    PROJECT_ROOT
    / "results"
    / "tables"
)

RESULTS_FIGURES = (
    PROJECT_ROOT
    / "results"
    / "figures"
)

RESULTS_LOGS = (
    PROJECT_ROOT
    / "results"
    / "logs"
)

RESULTS_MODELS = (
    PROJECT_ROOT
    / "results"
    / "models"
)

# ==================================================
# Input Dataset
# ==================================================

INPUT_DATA_PATH = (
    INTERIM_DATA
    / "county_year_vulnerability_scores.csv"
)

# ==================================================
# Output Folders
# ==================================================

RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS_LOGS.mkdir(parents=True, exist_ok=True)
RESULTS_MODELS.mkdir(parents=True, exist_ok=True)

# ==================================================
# Project Settings
# ==================================================

PROJECT_START_YEAR = 2011
PROJECT_END_YEAR = 2025

TRAIN_END_YEAR = 2022
TEST_START_YEAR = 2023

RANDOM_STATE = 42

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Notebook setup complete.")
print(f"Input dataset: {INPUT_DATA_PATH}")
print(f"Results tables folder: {RESULTS_TABLES}")
print(f"Results figures folder: {RESULTS_FIGURES}")
print(f"Results models folder: {RESULTS_MODELS}")

Notebook setup complete.
Input dataset: ..\data\interim\county_year_features\county_year_vulnerability_scores.csv
Results tables folder: ..\results\tables
Results figures folder: ..\results\figures
Results models folder: ..\results\models


## 2. Load Vulnerability-Ready Dataset

This section loads the final dataset created in Notebook 05.

This dataset already contains the raw county-year features, vulnerability sub-scores, final vulnerability score, and vulnerability class labels.

For this notebook, the vulnerability class will be used as the target variable, while leakage-related score columns will be removed before modelling.

In [6]:
# ==================================================
# Step 2: Load Vulnerability-Ready Dataset
# ==================================================

county_year_data = pd.read_csv(INPUT_DATA_PATH)

print("Vulnerability-ready dataset loaded successfully.")
print(f"Dataset shape: {county_year_data.shape}")

display(county_year_data.head())

Vulnerability-ready dataset loaded successfully.
Dataset shape: (1000, 123)


,STCOFIPS,RegionID,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,COUNTY,Year,avg_annual_housing_price,median_annual_housing_price,min_annual_housing_price,max_annual_housing_price,annual_price_volatility,monthly_observations,is_metro,SizeRank,POPULATION,CFLD_RISKS,HRCN_RISKS,SOVI_SCORE,RESL_SCORE,prev_year_housing_price,annual_price_growth_dollar,annual_price_growth_pct,baseline_housing_price,appreciation_from_baseline_pct,price_growth_acceleration,high_growth_flag,high_volatility_flag,complete_year_flag,neighbor_count,neighbor_avg_housing_price,neighbor_avg_price_growth_pct,neighbor_avg_price_volatility,neighbor_high_growth_share,neighbor_high_volatility_share,disaster_count,unique_disaster_events,climate_disaster_count,hurricane_disaster_count,tropical_storm_disaster_count,severe_storm_disaster_count,flood_disaster_count,fire_disaster_count,any_disaster_flag,any_climate_disaster_flag,cumulative_disaster_count,cumulative_climate_disaster_count,recent_3yr_disaster_count,recent_3yr_climate_disaster_count,acs_total_population,median_household_income,poverty_rate,unemployment_rate,owner_occupied_share,renter_occupied_share,median_gross_rent,acs_median_home_value,price_to_income_ratio,acs_estimated_2025_flag,nfip_claim_count,nfip_policy_count_sum,nfip_total_building_payment,nfip_total_contents_payment,nfip_total_icc_payment,nfip_total_claim_payment,nfip_avg_claim_payment,nfip_total_building_coverage,nfip_total_contents_coverage,nfip_cumulative_claim_count,nfip_cumulative_claim_payment,nfip_recent_3yr_claim_count,nfip_recent_3yr_claim_payment,nfip_claim_year_indicator,log_annual_price_volatility,log_price_to_income_ratio,log_median_gross_rent,log_neighbor_avg_price_volatility,log_nfip_claim_count,log_nfip_total_claim_payment,log_nfip_cumulative_claim_payment,log_nfip_recent_3yr_claim_payment,log_nfip_avg_claim_payment,scaled_annual_price_growth_pct,scaled_appreciation_from_baseline_pct,scaled_log_annual_price_volatility,scaled_price_growth_acceleration,scaled_log_price_to_income_ratio,scaled_log_median_gross_rent,scaled_median_household_income,scaled_CFLD_RISKS,scaled_HRCN_RISKS,scaled_climate_disaster_count,scaled_cumulative_climate_disaster_count,scaled_recent_3yr_climate_disaster_count,scaled_hurricane_disaster_count,scaled_SOVI_SCORE,scaled_poverty_rate,scaled_unemployment_rate,scaled_renter_occupied_share,scaled_RESL_SCORE,scaled_log_nfip_claim_count,scaled_log_nfip_total_claim_payment,scaled_log_nfip_cumulative_claim_payment,scaled_log_nfip_recent_3yr_claim_payment,scaled_log_nfip_avg_claim_payment,scaled_neighbor_avg_price_growth_pct,scaled_log_neighbor_avg_price_volatility,scaled_neighbor_high_growth_share,scaled_neighbor_high_volatility_share,scaled_low_income_stress,scaled_low_resilience,housing_pressure_score,affordability_stress_score,climate_exposure_score,disaster_history_score,socioeconomic_vulnerability_score,resilience_adjustment_score,insurance_loss_stress_score,spatial_spillover_score,climate_housing_vulnerability_score,vulnerability_class
0,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2011,"144,811.6070","144,342.9672","140,135.5499","151,065.3041","4,024.2825",12,1,251,277984,0.0000,96.7042,34.7646,80.9796,"162,126.8332","-17,315.2262",-10.6800,"162,126.8332",-10.6800,0.0000,0.0000,1,1,7,"102,374.8496",-6.7638,"1,794.8931",0.0000,0.1429,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"246,203.0000","41,373.0000",23.5571,7.3324,54.4678,45.5322,880.0000,"185,100.0000",3.5001,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,8.3004,1.5041,6.7811,7.4933,0.0000,0.0000,0.0000,0.0000,0.0000,0.0672,0.0310,0.5461,0.6198,0.3030,0.3665,0.1303,0.0000,0.8723,0.0000,0.0000,0.0000,0.0000,0.2702,0.6838,0.2453,0.9157,0.8400,0.0000,0.0000,0.0000,0.0000,0.0000,0.1196,0.2833,0.0000,0.1429,0.8697,0.1600,0.3160,0.5131,0.4362,0.0000,0.5287,0.1600,0.0000,0.1364,0.2613,Low
1,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2012,"136,832.4264","136,443.4221","134,664.5101","139

## 3. Validate Dataset Structure

This section checks the basic structure of the vulnerability-ready dataset before modelling.

In [7]:
# ==================================================
# Step 3: Validate Dataset Structure and Target Distribution
# ==================================================

# Basic structure checks
dataset_shape = county_year_data.shape
unique_counties = county_year_data["STCOFIPS"].nunique()
year_min = county_year_data["Year"].min()
year_max = county_year_data["Year"].max()

# Duplicate county-year rows
duplicate_county_year_rows = county_year_data.duplicated(
    subset=["STCOFIPS", "Year"]
).sum()

# Target distribution
target_distribution = county_year_data["vulnerability_class"].value_counts()
target_distribution_pct = county_year_data["vulnerability_class"].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    "count": target_distribution,
    "percentage": target_distribution_pct
})

# Missing values in target
target_missing = county_year_data["vulnerability_class"].isna().sum()

print("Dataset validation summary:")
print(f"Dataset shape: {dataset_shape}")
print(f"Unique counties: {unique_counties}")
print(f"Year range: {year_min} to {year_max}")
print(f"Duplicate county-year rows: {duplicate_county_year_rows}")
print(f"Missing target values: {target_missing}")

print("\nTarget class distribution:")
display(target_summary)

Dataset validation summary:
Dataset shape: (1000, 123)
Unique counties: 67
Year range: 2011 to 2025
Duplicate county-year rows: 0
Missing target values: 0

Target class distribution:


,count,percentage
vulnerability_class,,
Low,334,33.4000
Medium,333,33.3000
High,333,33.3000


## 4. Define Target Variable

This section defines `vulnerability_class` as the target variable for supervised classification.

In [8]:
# ==================================================
# Step 4: Define Target Variable
# ==================================================

target_column = "vulnerability_class"

y = county_year_data[target_column].copy()

print(f"Target variable: {target_column}")
print(f"Number of target observations: {len(y)}")

print("\nTarget distribution:")
display(y.value_counts())

Target variable: vulnerability_class
Number of target observations: 1000

Target distribution:


vulnerability_class
Low       334
Medium    333
High      333
Name: count, dtype: int64

## 5. Identify Leakage Columns

This section identifies columns that should not be used as model inputs because they directly come from the vulnerability score construction.

In [9]:
# ==================================================
# Step 5: Identify Leakage Columns
# ==================================================

# Final target and score columns
target_and_final_score_columns = [
    "vulnerability_class",
    "climate_housing_vulnerability_score"
]

# Vulnerability sub-score columns created in Notebook 05
sub_score_columns = [
    "housing_pressure_score",
    "affordability_stress_score",
    "climate_exposure_score",
    "disaster_history_score",
    "socioeconomic_vulnerability_score",
    "resilience_adjustment_score",
    "insurance_loss_stress_score",
    "spatial_spillover_score"
]

# Scaled variables created only for score construction
scaled_columns = [
    col for col in county_year_data.columns
    if col.startswith("scaled_")
]

# Log helper variables created only for score construction
log_helper_columns = [
    col for col in county_year_data.columns
    if col.startswith("log_")
]

# Identifier / metadata columns that should not be used as predictors
metadata_columns = [
    "STCOFIPS",
    "RegionID",
    "RegionName",
    "State",
    "Metro",
    "StateCodeFIPS",
    "MunicipalCodeFIPS",
    "COUNTY"
]

# Combine all columns to exclude from modelling
leakage_and_excluded_columns = (
    target_and_final_score_columns
    + sub_score_columns
    + scaled_columns
    + log_helper_columns
    + metadata_columns
)

# Keep only columns that actually exist in the dataset
leakage_and_excluded_columns = [
    col for col in leakage_and_excluded_columns
    if col in county_year_data.columns
]

print(f"Total excluded columns: {len(leakage_and_excluded_columns)}")

print("\nExcluded columns:")
for col in leakage_and_excluded_columns:
    print(f"- {col}")

Total excluded columns: 56

Excluded columns:
- vulnerability_class
- climate_housing_vulnerability_score
- housing_pressure_score
- affordability_stress_score
- climate_exposure_score
- disaster_history_score
- socioeconomic_vulnerability_score
- resilience_adjustment_score
- insurance_loss_stress_score
- spatial_spillover_score
- scaled_annual_price_growth_pct
- scaled_appreciation_from_baseline_pct
- scaled_log_annual_price_volatility
- scaled_price_growth_acceleration
- scaled_log_price_to_income_ratio
- scaled_log_median_gross_rent
- scaled_median_household_income
- scaled_CFLD_RISKS
- scaled_HRCN_RISKS
- scaled_climate_disaster_count
- scaled_cumulative_climate_disaster_count
- scaled_recent_3yr_climate_disaster_count
- scaled_hurricane_disaster_count
- scaled_SOVI_SCORE
- scaled_poverty_rate
- scaled_unemployment_rate
- scaled_renter_occupied_share
- scaled_RESL_SCORE
- scaled_log_nfip_claim_count
- scaled_log_nfip_total_claim_payment
- scaled_log_nfip_cumulative_claim_payment
-

## 6. Define Feature Matrix

This section creates the predictor dataset after removing leakage columns and non-numeric columns.

In [10]:
# ==================================================
# Step 6: Define Feature Matrix
# ==================================================

# Start from all columns except leakage and excluded columns
candidate_feature_columns = [
    col for col in county_year_data.columns
    if col not in leakage_and_excluded_columns
]

# Keep only numeric columns for modelling
numeric_feature_columns = county_year_data[candidate_feature_columns].select_dtypes(
    include=["number"]
).columns.tolist()

# Create feature matrix
X = county_year_data[numeric_feature_columns].copy()

# Missing values in selected model features
missing_feature_values = X.isna().sum()
missing_feature_values = missing_feature_values[missing_feature_values > 0].sort_values(ascending=False)

print(f"Candidate feature columns before numeric filtering: {len(candidate_feature_columns)}")
print(f"Final numeric model features: {len(numeric_feature_columns)}")

print("\nSelected feature columns:")
for col in numeric_feature_columns:
    print(f"- {col}")

print("\nMissing values in selected features:")
if len(missing_feature_values) == 0:
    print("No missing values in selected model features.")
else:
    display(missing_feature_values)

Candidate feature columns before numeric filtering: 67
Final numeric model features: 67

Selected feature columns:
- Year
- avg_annual_housing_price
- median_annual_housing_price
- min_annual_housing_price
- max_annual_housing_price
- annual_price_volatility
- monthly_observations
- is_metro
- SizeRank
- POPULATION
- CFLD_RISKS
- HRCN_RISKS
- SOVI_SCORE
- RESL_SCORE
- prev_year_housing_price
- annual_price_growth_dollar
- annual_price_growth_pct
- baseline_housing_price
- appreciation_from_baseline_pct
- price_growth_acceleration
- high_growth_flag
- high_volatility_flag
- complete_year_flag
- neighbor_count
- neighbor_avg_housing_price
- neighbor_avg_price_growth_pct
- neighbor_avg_price_volatility
- neighbor_high_growth_share
- neighbor_high_volatility_share
- disaster_count
- unique_disaster_events
- climate_disaster_count
- hurricane_disaster_count
- tropical_storm_disaster_count
- severe_storm_disaster_count
- flood_disaster_count
- fire_disaster_count
- any_disaster_flag
- any_cl

prev_year_housing_price       3
annual_price_growth_dollar    3
high_growth_flag              3
dtype: int64

## 7. Temporal Train-Test Split

This section splits the data into earlier training years and later test years.

In [11]:
# ==================================================
# Step 7: Temporal Train-Test Split
# ==================================================

# Sort data by year and county before splitting
county_year_data_sorted = county_year_data.sort_values(
    ["Year", "STCOFIPS"]
).reset_index(drop=True)

# Re-create X and y using the sorted dataset
X_sorted = county_year_data_sorted[numeric_feature_columns].copy()
y_sorted = county_year_data_sorted[target_column].copy()

# Temporal split
train_mask = county_year_data_sorted["Year"] <= TRAIN_END_YEAR
test_mask = county_year_data_sorted["Year"] >= TEST_START_YEAR

X_train = X_sorted.loc[train_mask].copy()
y_train = y_sorted.loc[train_mask].copy()

X_test = X_sorted.loc[test_mask].copy()
y_test = y_sorted.loc[test_mask].copy()

train_year_range = (
    county_year_data_sorted.loc[train_mask, "Year"].min(),
    county_year_data_sorted.loc[train_mask, "Year"].max()
)

test_year_range = (
    county_year_data_sorted.loc[test_mask, "Year"].min(),
    county_year_data_sorted.loc[test_mask, "Year"].max()
)

print("Temporal train-test split complete.")
print(f"Training years: {train_year_range[0]} to {train_year_range[1]}")
print(f"Testing years: {test_year_range[0]} to {test_year_range[1]}")

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

print("\nTraining target distribution:")
display(y_train.value_counts())

print("\nTesting target distribution:")
display(y_test.value_counts())

Temporal train-test split complete.
Training years: 2011 to 2022
Testing years: 2023 to 2025

X_train shape: (799, 67)
X_test shape: (201, 67)

Training target distribution:


vulnerability_class
Low       281
High      264
Medium    254
Name: count, dtype: int64


Testing target distribution:


vulnerability_class
Medium    79
High      69
Low       53
Name: count, dtype: int64

In [12]:
# ==================================================
# Confirm Final Supervised-Model Predictor List
# ==================================================

final_predictors = X_train.columns.tolist()

print(f"Total predictors: {len(final_predictors)}\n")

for number, predictor in enumerate(final_predictors, start=1):
    print(f"{number:02d}. {predictor}")

Total predictors: 67

01. Year
02. avg_annual_housing_price
03. median_annual_housing_price
04. min_annual_housing_price
05. max_annual_housing_price
06. annual_price_volatility
07. monthly_observations
08. is_metro
09. SizeRank
10. POPULATION
11. CFLD_RISKS
12. HRCN_RISKS
13. SOVI_SCORE
14. RESL_SCORE
15. prev_year_housing_price
16. annual_price_growth_dollar
17. annual_price_growth_pct
18. baseline_housing_price
19. appreciation_from_baseline_pct
20. price_growth_acceleration
21. high_growth_flag
22. high_volatility_flag
23. complete_year_flag
24. neighbor_count
25. neighbor_avg_housing_price
26. neighbor_avg_price_growth_pct
27. neighbor_avg_price_volatility
28. neighbor_high_growth_share
29. neighbor_high_volatility_share
30. disaster_count
31. unique_disaster_events
32. climate_disaster_count
33. hurricane_disaster_count
34. tropical_storm_disaster_count
35. severe_storm_disaster_count
36. flood_disaster_count
37. fire_disaster_count
38. any_disaster_flag
39. any_climate_disaster_

## 8. Create Year-Based Cross-Validation Folds

This section creates temporal cross-validation folds using only the training years.

In [13]:
# ==================================================
# Step 8: Create Year-Based Cross-Validation Folds
# ==================================================

cv_year_folds = [
    {
        "fold": 1,
        "train_years": list(range(2011, 2015)),
        "validation_years": list(range(2015, 2017))
    },
    {
        "fold": 2,
        "train_years": list(range(2011, 2017)),
        "validation_years": list(range(2017, 2019))
    },
    {
        "fold": 3,
        "train_years": list(range(2011, 2019)),
        "validation_years": list(range(2019, 2021))
    },
    {
        "fold": 4,
        "train_years": list(range(2011, 2021)),
        "validation_years": [2021]
    },
    {
        "fold": 5,
        "train_years": list(range(2011, 2022)),
        "validation_years": [2022]
    }
]

cv_summary_rows = []

for fold_info in cv_year_folds:
    fold_number = fold_info["fold"]
    train_years = fold_info["train_years"]
    validation_years = fold_info["validation_years"]
    
    fold_train_mask = county_year_data_sorted["Year"].isin(train_years)
    fold_validation_mask = county_year_data_sorted["Year"].isin(validation_years)
    
    cv_summary_rows.append({
        "fold": fold_number,
        "train_years": f"{min(train_years)}-{max(train_years)}",
        "validation_years": f"{min(validation_years)}-{max(validation_years)}",
        "train_rows": fold_train_mask.sum(),
        "validation_rows": fold_validation_mask.sum()
    })

cv_summary = pd.DataFrame(cv_summary_rows)

print("Year-based cross-validation folds created.")
display(cv_summary)

Year-based cross-validation folds created.


,fold,train_years,validation_years,train_rows,validation_rows
0,1,2011-2014,2015-2016,264,133
1,2,2011-2016,2017-2018,397,134
2,3,2011-2018,2019-2020,531,134
3,4,2011-2020,2021-2021,665,67
4,5,2011-2021,2022-2022,732,67


## 9. Define Model Evaluation Functions

This section creates helper functions for cross-validation and final test evaluation.

In [14]:
# ==================================================
# Step 9: Define Model Evaluation Functions
# ==================================================

def calculate_classification_metrics(y_true, y_pred):
    """
    Calculate the main classification metrics used in this notebook.
    """
    
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }
    
    return metrics


def run_year_based_cv(model_name, model_pipeline):
    """
    Run year-based temporal cross-validation using the folds defined earlier.
    """
    
    cv_results = []
    
    for fold_info in cv_year_folds:
        fold_number = fold_info["fold"]
        train_years = fold_info["train_years"]
        validation_years = fold_info["validation_years"]
        
        fold_train_mask = county_year_data_sorted["Year"].isin(train_years)
        fold_validation_mask = county_year_data_sorted["Year"].isin(validation_years)
        
        X_fold_train = X_sorted.loc[fold_train_mask, numeric_feature_columns].copy()
        y_fold_train = y_sorted.loc[fold_train_mask].copy()
        
        X_fold_validation = X_sorted.loc[fold_validation_mask, numeric_feature_columns].copy()
        y_fold_validation = y_sorted.loc[fold_validation_mask].copy()
        
        model_pipeline.fit(X_fold_train, y_fold_train)
        y_fold_pred = model_pipeline.predict(X_fold_validation)
        
        fold_metrics = calculate_classification_metrics(
            y_fold_validation,
            y_fold_pred
        )
        
        fold_metrics["model"] = model_name
        fold_metrics["fold"] = fold_number
        fold_metrics["train_years"] = f"{min(train_years)}-{max(train_years)}"
        fold_metrics["validation_years"] = f"{min(validation_years)}-{max(validation_years)}"
        
        cv_results.append(fold_metrics)
    
    cv_results_df = pd.DataFrame(cv_results)
    
    return cv_results_df


def evaluate_on_test_set(model_name, model_pipeline):
    """
    Fit the model on the full training period and evaluate on the holdout test period.
    """
    
    model_pipeline.fit(X_train, y_train)
    y_test_pred = model_pipeline.predict(X_test)
    
    test_metrics = calculate_classification_metrics(
        y_test,
        y_test_pred
    )
    
    test_metrics["model"] = model_name
    
    test_predictions = pd.DataFrame({
        "Year": county_year_data_sorted.loc[test_mask, "Year"].values,
        "STCOFIPS": county_year_data_sorted.loc[test_mask, "STCOFIPS"].values,
        "RegionName": county_year_data_sorted.loc[test_mask, "RegionName"].values,
        "actual_vulnerability_class": y_test.values,
        "predicted_vulnerability_class": y_test_pred
    })
    
    return test_metrics, test_predictions, model_pipeline


print("Model evaluation functions created.")

Model evaluation functions created.


## 10. Naive Baseline Model

This section creates a simple reference model based on the most common class in the training data.

In [15]:
# ==================================================
# Step 10: Naive Baseline Model
# ==================================================

naive_baseline_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

# Cross-validation performance
naive_baseline_cv_results = run_year_based_cv(
    model_name="Naive Baseline",
    model_pipeline=naive_baseline_model
)

# Final test performance
naive_baseline_test_metrics, naive_baseline_test_predictions, naive_baseline_fitted_model = evaluate_on_test_set(
    model_name="Naive Baseline",
    model_pipeline=naive_baseline_model
)

print("Naive Baseline cross-validation results:")
display(naive_baseline_cv_results)

print("\nNaive Baseline test metrics:")
display(pd.DataFrame([naive_baseline_test_metrics]))

Naive Baseline cross-validation results:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model,fold,train_years,validation_years
0,0.3910,0.1303,0.3333,0.1874,0.2198,Naive Baseline,1,2011-2014,2015-2016
1,0.2015,0.0672,0.3333,0.1118,0.0676,Naive Baseline,2,2011-2016,2017-2018
2,0.2388,0.0796,0.3333,0.1285,0.0921,Naive Baseline,3,2011-2018,2019-2020
3,0.0448,0.0149,0.3333,0.0286,0.0038,Naive Baseline,4,2011-2020,2021-2021
4,0.0149,0.0050,0.3333,0.0098,0.0004,Naive Baseline,5,2011-2021,2022-2022



Naive Baseline test metrics:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model
0,0.2637,0.0879,0.3333,0.1391,0.1100,Naive Baseline


## 11. Logistic Regression Baseline Model

This section trains a multinomial logistic regression model as the main statistical baseline.

In [16]:
# ==================================================
# Step 11: Logistic Regression Baseline Model
# ==================================================

logistic_regression_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

# Cross-validation performance
logistic_regression_cv_results = run_year_based_cv(
    model_name="Logistic Regression",
    model_pipeline=logistic_regression_model
)

# Final test performance
logistic_regression_test_metrics, logistic_regression_test_predictions, logistic_regression_fitted_model = evaluate_on_test_set(
    model_name="Logistic Regression",
    model_pipeline=logistic_regression_model
)

print("Logistic Regression cross-validation results:")
display(logistic_regression_cv_results)

print("\nLogistic Regression test metrics:")
display(pd.DataFrame([logistic_regression_test_metrics]))

Logistic Regression cross-validation results:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model,fold,train_years,validation_years
0,0.7669,0.7751,0.7479,0.7530,0.7698,Logistic Regression,1,2011-2014,2015-2016
1,0.6866,0.7564,0.6420,0.6631,0.6703,Logistic Regression,2,2011-2016,2017-2018
2,0.7836,0.8258,0.7887,0.7890,0.7765,Logistic Regression,3,2011-2018,2019-2020
3,0.7164,0.8321,0.5511,0.5927,0.6709,Logistic Regression,4,2011-2020,2021-2021
4,0.9254,0.8460,0.9222,0.8746,0.9319,Logistic Regression,5,2011-2021,2022-2022



Logistic Regression test metrics:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model
0,0.7761,0.8038,0.8077,0.7687,0.7622,Logistic Regression


The Logistic Regression model performed substantially better than the naive baseline. On the 2023–2025 holdout test set, it achieved an accuracy of 0.776 and a macro F1 score of 0.769. This suggests that the raw county-year features contain meaningful information for distinguishing Low, Medium, and High vulnerability classes. The cross-validation results also show some variation across time periods, which supports the use of temporal validation instead of a random split.

## 12. Decision Tree Classifier

This section trains a decision tree model to classify vulnerability levels using nonlinear decision rules.

In [17]:
# ==================================================
# Step 12: Decision Tree Classifier
# ==================================================

decision_tree_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=10,
        random_state=RANDOM_STATE
    ))
])

# Cross-validation performance
decision_tree_cv_results = run_year_based_cv(
    model_name="Decision Tree",
    model_pipeline=decision_tree_model
)

# Final test performance
decision_tree_test_metrics, decision_tree_test_predictions, decision_tree_fitted_model = evaluate_on_test_set(
    model_name="Decision Tree",
    model_pipeline=decision_tree_model
)

print("Decision Tree cross-validation results:")
display(decision_tree_cv_results)

print("\nDecision Tree test metrics:")
display(pd.DataFrame([decision_tree_test_metrics]))

Decision Tree cross-validation results:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model,fold,train_years,validation_years
0,0.6090,0.7324,0.6039,0.6160,0.6117,Decision Tree,1,2011-2014,2015-2016
1,0.6343,0.6555,0.6625,0.6312,0.6460,Decision Tree,2,2011-2016,2017-2018
2,0.6269,0.6371,0.6051,0.6120,0.6220,Decision Tree,3,2011-2018,2019-2020
3,0.6567,0.5979,0.7699,0.5968,0.6799,Decision Tree,4,2011-2020,2021-2021
4,0.6716,0.3919,0.5000,0.3716,0.7382,Decision Tree,5,2011-2021,2022-2022



Decision Tree test metrics:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model
0,0.5473,0.5575,0.5787,0.5295,0.5175,Decision Tree


The Decision Tree model performed better than the naive baseline but worse than Logistic Regression on the holdout test set. This suggests that a single shallow tree may not capture the full structure of the vulnerability classification problem. The result supports testing ensemble models such as Random Forest, which can reduce the instability of individual trees.

## 13. Random Forest Classifier

This section trains a Random Forest model using an ensemble of decision trees.

In [18]:
# ==================================================
# Step 13: Random Forest Classifier
# ==================================================

random_forest_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Cross-validation performance
random_forest_cv_results = run_year_based_cv(
    model_name="Random Forest",
    model_pipeline=random_forest_model
)

# Final test performance
random_forest_test_metrics, random_forest_test_predictions, random_forest_fitted_model = evaluate_on_test_set(
    model_name="Random Forest",
    model_pipeline=random_forest_model
)

print("Random Forest cross-validation results:")
display(random_forest_cv_results)

print("\nRandom Forest test metrics:")
display(pd.DataFrame([random_forest_test_metrics]))

Random Forest cross-validation results:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model,fold,train_years,validation_years
0,0.7970,0.8383,0.7687,0.7777,0.7891,Random Forest,1,2011-2014,2015-2016
1,0.6940,0.7124,0.7222,0.7054,0.6980,Random Forest,2,2011-2016,2017-2018
2,0.7910,0.8238,0.7770,0.7913,0.7930,Random Forest,3,2011-2018,2019-2020
3,0.7612,0.7305,0.8489,0.7571,0.7658,Random Forest,4,2011-2020,2021-2021
4,0.7910,0.5926,0.8722,0.6539,0.8340,Random Forest,5,2011-2021,2022-2022



Random Forest test metrics:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model
0,0.6418,0.6445,0.6708,0.6321,0.6159,Random Forest


The Random Forest model improved over the single Decision Tree, but it did not outperform Logistic Regression on the 2023–2025 holdout test set. This suggests that the vulnerability classes may be captured well by relatively stable linear patterns across the selected county-year features. However, the Random Forest remains useful because it provides tree-based feature importance and can support later explainability analysis.

## 14. XGBoost Classifier

This section trains an XGBoost classifier if the package is available in the current environment.

In [19]:
# ==================================================
# Step 14: XGBoost Classifier
# ==================================================

try:
    from xgboost import XGBClassifier
    
    # Encode class labels for XGBoost
    label_encoder = LabelEncoder()
    y_train_encoded = pd.Series(
        label_encoder.fit_transform(y_train),
        index=y_train.index
    )
    y_test_encoded = pd.Series(
        label_encoder.transform(y_test),
        index=y_test.index
    )
    y_sorted_encoded = pd.Series(
        label_encoder.transform(y_sorted),
        index=y_sorted.index
    )
    
    xgboost_model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softmax",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE
        ))
    ])
    
    xgboost_cv_results = []
    
    for fold_info in cv_year_folds:
        fold_number = fold_info["fold"]
        train_years = fold_info["train_years"]
        validation_years = fold_info["validation_years"]
        
        fold_train_mask = county_year_data_sorted["Year"].isin(train_years)
        fold_validation_mask = county_year_data_sorted["Year"].isin(validation_years)
        
        X_fold_train = X_sorted.loc[fold_train_mask, numeric_feature_columns].copy()
        y_fold_train = y_sorted_encoded.loc[fold_train_mask].copy()
        
        X_fold_validation = X_sorted.loc[fold_validation_mask, numeric_feature_columns].copy()
        y_fold_validation = y_sorted_encoded.loc[fold_validation_mask].copy()
        
        xgboost_model.fit(X_fold_train, y_fold_train)
        y_fold_pred_encoded = xgboost_model.predict(X_fold_validation)
        
        y_fold_validation_labels = label_encoder.inverse_transform(y_fold_validation)
        y_fold_pred_labels = label_encoder.inverse_transform(y_fold_pred_encoded)
        
        fold_metrics = calculate_classification_metrics(
            y_fold_validation_labels,
            y_fold_pred_labels
        )
        
        fold_metrics["model"] = "XGBoost"
        fold_metrics["fold"] = fold_number
        fold_metrics["train_years"] = f"{min(train_years)}-{max(train_years)}"
        fold_metrics["validation_years"] = f"{min(validation_years)}-{max(validation_years)}"
        
        xgboost_cv_results.append(fold_metrics)
    
    xgboost_cv_results = pd.DataFrame(xgboost_cv_results)
    
    # Final test evaluation
    xgboost_model.fit(X_train, y_train_encoded)
    y_test_pred_encoded = xgboost_model.predict(X_test)
    y_test_pred_labels = label_encoder.inverse_transform(y_test_pred_encoded)
    
    xgboost_test_metrics = calculate_classification_metrics(
        y_test,
        y_test_pred_labels
    )
    xgboost_test_metrics["model"] = "XGBoost"
    
    xgboost_test_predictions = pd.DataFrame({
        "Year": county_year_data_sorted.loc[test_mask, "Year"].values,
        "STCOFIPS": county_year_data_sorted.loc[test_mask, "STCOFIPS"].values,
        "RegionName": county_year_data_sorted.loc[test_mask, "RegionName"].values,
        "actual_vulnerability_class": y_test.values,
        "predicted_vulnerability_class": y_test_pred_labels
    })
    
    xgboost_fitted_model = xgboost_model
    
    print("XGBoost cross-validation results:")
    display(xgboost_cv_results)
    
    print("\nXGBoost test metrics:")
    display(pd.DataFrame([xgboost_test_metrics]))

except ImportError:
    xgboost_cv_results = None
    xgboost_test_metrics = None
    xgboost_test_predictions = None
    xgboost_fitted_model = None
    
    print("XGBoost is not installed in this environment.")
    print("Skipping XGBoost model.")

XGBoost cross-validation results:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model,fold,train_years,validation_years
0,0.8195,0.8564,0.7998,0.8113,0.8184,XGBoost,1,2011-2014,2015-2016
1,0.7388,0.7506,0.7532,0.7462,0.7422,XGBoost,2,2011-2016,2017-2018
2,0.8433,0.8398,0.8457,0.8416,0.8415,XGBoost,3,2011-2018,2019-2020
3,0.7761,0.7073,0.8526,0.7423,0.7842,XGBoost,4,2011-2020,2021-2021
4,0.7910,0.5926,0.8722,0.6539,0.8340,XGBoost,5,2011-2021,2022-2022



XGBoost test metrics:


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,model
0,0.6468,0.6543,0.6887,0.6221,0.6033,XGBoost


XGBoost achieved strong cross-validation performance but did not outperform Logistic Regression on the 2023–2025 holdout test set. This suggests that the more complex boosted-tree model may be less stable when applied to later years. Logistic Regression remains the strongest model so far based on holdout macro F1.

## 15. Hyperparameter Tuning for Tree-Based Models

Optuna was used to tune Random Forest and XGBoost to check whether tree-based model performance improved after systematic hyperparameter tuning.

In [20]:
# ==================================================
# Step 15: Hyperparameter Tuning for Tree-Based Models
# ==================================================

import optuna
import warnings

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

optuna_model_outputs = {}
optuna_tuning_summary_rows = []

N_OPTUNA_TRIALS = 25

print("Optuna setup completed.")

# --------------------------------------------------
# Helper function: CV macro F1 using year-based folds
# --------------------------------------------------

def run_optuna_cv_score(model_pipeline, use_encoded_target=False, encoder=None):
    """
    Evaluate one Optuna trial using the existing year-based CV folds.
    Returns mean validation macro F1.
    """
    
    fold_macro_f1_scores = []
    
    for fold_info in cv_year_folds:
        train_years = fold_info["train_years"]
        validation_years = fold_info["validation_years"]
        
        fold_train_mask = county_year_data_sorted["Year"].isin(train_years)
        fold_validation_mask = county_year_data_sorted["Year"].isin(validation_years)
        
        X_fold_train = X_sorted.loc[fold_train_mask, numeric_feature_columns].copy()
        y_fold_train = y_sorted.loc[fold_train_mask].copy()
        
        X_fold_validation = X_sorted.loc[fold_validation_mask, numeric_feature_columns].copy()
        y_fold_validation = y_sorted.loc[fold_validation_mask].copy()
        
        if use_encoded_target:
            y_fold_train_model = encoder.transform(y_fold_train)
            
            model_pipeline.fit(X_fold_train, y_fold_train_model)
            y_fold_pred_encoded = model_pipeline.predict(X_fold_validation)
            y_fold_pred = encoder.inverse_transform(y_fold_pred_encoded.astype(int))
        
        else:
            model_pipeline.fit(X_fold_train, y_fold_train)
            y_fold_pred = model_pipeline.predict(X_fold_validation)
        
        fold_macro_f1 = f1_score(
            y_fold_validation,
            y_fold_pred,
            average="macro",
            zero_division=0
        )
        
        fold_macro_f1_scores.append(fold_macro_f1)
    
    return np.mean(fold_macro_f1_scores)


# --------------------------------------------------
# Helper function: evaluate tuned model on holdout test set
# --------------------------------------------------

def evaluate_optuna_model_on_test(model_name, model_pipeline, use_encoded_target=False, encoder=None):
    """
    Fit the tuned model on the full training period and evaluate it on the 2023–2025 holdout test set.
    """
    
    if use_encoded_target:
        y_train_model = encoder.transform(y_train)
        
        model_pipeline.fit(X_train, y_train_model)
        y_test_pred_encoded = model_pipeline.predict(X_test)
        y_test_pred = encoder.inverse_transform(y_test_pred_encoded.astype(int))
    
    else:
        model_pipeline.fit(X_train, y_train)
        y_test_pred = model_pipeline.predict(X_test)
    
    test_metrics = calculate_classification_metrics(
        y_test,
        y_test_pred
    )
    
    test_metrics["model"] = model_name
    
    test_predictions = pd.DataFrame({
        "Year": county_year_data_sorted.loc[test_mask, "Year"].values,
        "STCOFIPS": county_year_data_sorted.loc[test_mask, "STCOFIPS"].values,
        "RegionName": county_year_data_sorted.loc[test_mask, "RegionName"].values,
        "actual_vulnerability_class": y_test.values,
        "predicted_vulnerability_class": y_test_pred
    })
    
    return test_metrics, test_predictions, model_pipeline

# --------------------------------------------------
# Helper function: full CV results for tuned Optuna models
# --------------------------------------------------

def run_optuna_full_cv_results(model_name, model_pipeline, use_encoded_target=False, encoder=None):
    """
    Runs full year-based cross-validation for the tuned Optuna model.
    Returns fold-level accuracy, macro F1, weighted F1, and related metrics.
    """
    
    tuned_cv_results = []
    
    for fold_info in cv_year_folds:
        fold_number = fold_info["fold"]
        train_years = fold_info["train_years"]
        validation_years = fold_info["validation_years"]
        
        fold_train_mask = county_year_data_sorted["Year"].isin(train_years)
        fold_validation_mask = county_year_data_sorted["Year"].isin(validation_years)
        
        X_fold_train = X_sorted.loc[fold_train_mask, numeric_feature_columns].copy()
        y_fold_train = y_sorted.loc[fold_train_mask].copy()
        
        X_fold_validation = X_sorted.loc[fold_validation_mask, numeric_feature_columns].copy()
        y_fold_validation = y_sorted.loc[fold_validation_mask].copy()
        
        if use_encoded_target:
            y_fold_train_model = encoder.transform(y_fold_train)
            
            model_pipeline.fit(X_fold_train, y_fold_train_model)
            y_fold_pred_encoded = model_pipeline.predict(X_fold_validation)
            y_fold_pred = encoder.inverse_transform(y_fold_pred_encoded.astype(int))
        
        else:
            model_pipeline.fit(X_fold_train, y_fold_train)
            y_fold_pred = model_pipeline.predict(X_fold_validation)
        
        fold_metrics = calculate_classification_metrics(
            y_fold_validation,
            y_fold_pred
        )
        
        fold_metrics["model"] = model_name
        fold_metrics["fold"] = fold_number
        fold_metrics["train_years"] = f"{min(train_years)}-{max(train_years)}"
        fold_metrics["validation_years"] = f"{min(validation_years)}-{max(validation_years)}"
        
        tuned_cv_results.append(fold_metrics)
    
    return pd.DataFrame(tuned_cv_results)

Optuna setup completed.


In [21]:
# ==================================================
# Step 15.1: Tune Random Forest with Optuna
# ==================================================

def create_optuna_random_forest_pipeline(params):
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])


def random_forest_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=100),
        "max_depth": trial.suggest_categorical("max_depth", [4, 6, 8, 10, 12, None]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    }
    
    model_pipeline = create_optuna_random_forest_pipeline(params)
    
    return run_optuna_cv_score(
        model_pipeline=model_pipeline,
        use_encoded_target=False
    )


random_forest_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

random_forest_study.optimize(
    random_forest_objective,
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=False
)

optuna_random_forest_model = create_optuna_random_forest_pipeline(
    random_forest_study.best_params
)

optuna_random_forest_cv_results = run_optuna_full_cv_results(
    model_name="Random Forest (Optuna)",
    model_pipeline=optuna_random_forest_model,
    use_encoded_target=False
)

(
    optuna_random_forest_test_metrics,
    optuna_random_forest_test_predictions,
    optuna_random_forest_fitted_model
) = evaluate_optuna_model_on_test(
    model_name="Random Forest (Optuna)",
    model_pipeline=optuna_random_forest_model
)

optuna_model_outputs["Random Forest (Optuna)"] = {
    "cv_results": optuna_random_forest_cv_results,
    "test_metrics": optuna_random_forest_test_metrics,
    "test_predictions": optuna_random_forest_test_predictions,
    "fitted_model": optuna_random_forest_fitted_model,
    "best_params": random_forest_study.best_params,
    "best_cv_macro_f1": random_forest_study.best_value
}

optuna_tuning_summary_rows.append({
    "model": "Random Forest (Optuna)",
    "best_cv_macro_f1": random_forest_study.best_value,
    "holdout_accuracy": optuna_random_forest_test_metrics["accuracy"],
    "holdout_macro_f1": optuna_random_forest_test_metrics["macro_f1"],
    "best_params": random_forest_study.best_params
})

print("Random Forest tuning completed.")

Random Forest tuning completed.


In [22]:
# ==================================================
# Step 15.2: Tune XGBoost with Optuna
# ==================================================

optuna_label_encoder = LabelEncoder()
optuna_label_encoder.fit(y_train)


def create_optuna_xgboost_pipeline(params):
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", XGBClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            min_child_weight=params["min_child_weight"],
            gamma=params["gamma"],
            reg_alpha=params["reg_alpha"],
            reg_lambda=params["reg_lambda"],
            objective="multi:softmax",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])


def xgboost_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=100),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.001, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.001, 10.0, log=True)
    }
    
    model_pipeline = create_optuna_xgboost_pipeline(params)
    
    return run_optuna_cv_score(
        model_pipeline=model_pipeline,
        use_encoded_target=True,
        encoder=optuna_label_encoder
    )


xgboost_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

xgboost_study.optimize(
    xgboost_objective,
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=False
)

optuna_xgboost_model = create_optuna_xgboost_pipeline(
    xgboost_study.best_params
)

optuna_xgboost_cv_results = run_optuna_full_cv_results(
    model_name="XGBoost (Optuna)",
    model_pipeline=optuna_xgboost_model,
    use_encoded_target=True,
    encoder=optuna_label_encoder
)

(
    optuna_xgboost_test_metrics,
    optuna_xgboost_test_predictions,
    optuna_xgboost_fitted_model
) = evaluate_optuna_model_on_test(
    model_name="XGBoost (Optuna)",
    model_pipeline=optuna_xgboost_model,
    use_encoded_target=True,
    encoder=optuna_label_encoder
)

optuna_model_outputs["XGBoost (Optuna)"] = {
    "cv_results": optuna_xgboost_cv_results,
    "test_metrics": optuna_xgboost_test_metrics,
    "test_predictions": optuna_xgboost_test_predictions,
    "fitted_model": optuna_xgboost_fitted_model,
    "best_params": xgboost_study.best_params,
    "best_cv_macro_f1": xgboost_study.best_value
}

optuna_tuning_summary_rows.append({
    "model": "XGBoost (Optuna)",
    "best_cv_macro_f1": xgboost_study.best_value,
    "holdout_accuracy": optuna_xgboost_test_metrics["accuracy"],
    "holdout_macro_f1": optuna_xgboost_test_metrics["macro_f1"],
    "best_params": xgboost_study.best_params
})

print("XGBoost tuning completed.")

XGBoost tuning completed.


In [31]:
print("Random Forest best parameters:")
print(random_forest_study.best_params)

print("\nRandom Forest best CV macro F1:")
print(random_forest_study.best_value)

print("\nXGBoost best parameters:")
print(xgboost_study.best_params)

print("\nXGBoost best CV macro F1:")
print(xgboost_study.best_value)

Random Forest best parameters:
{'n_estimators': 600, 'max_depth': 8, 'min_samples_split': 14, 'min_samples_leaf': 5, 'max_features': 'log2'}

Random Forest best CV macro F1:
0.746855152077098

XGBoost best parameters:
{'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.14375148048694283, 'subsample': 0.9647749916436704, 'colsample_bytree': 0.9429178011056337, 'min_child_weight': 7, 'gamma': 0.6911407800788922, 'reg_alpha': 0.007017110749133235, 'reg_lambda': 0.028912644384523085}

XGBoost best CV macro F1:
0.7933791982010554


In [23]:
# ==================================================
# Step 15.3: Optuna Tuning Summary
# ==================================================

optuna_tuning_summary = pd.DataFrame(optuna_tuning_summary_rows)

optuna_tuning_summary = optuna_tuning_summary[
    [
        "model",
        "best_cv_macro_f1",
        "holdout_accuracy",
        "holdout_macro_f1",
        "best_params"
    ]
].sort_values(
    by="holdout_macro_f1",
    ascending=False
).reset_index(drop=True)

print("Optuna tuning summary:")
display(optuna_tuning_summary)

Optuna tuning summary:


,model,best_cv_macro_f1,holdout_accuracy,holdout_macro_f1,best_params
0,XGBoost (Optuna),0.7934,0.7313,0.7218,"{'n_estimators': 500, 'max_depth': 2, 'learnin..."
1,Random Forest (Optuna),0.7469,0.6169,0.6029,"{'n_estimators': 600, 'max_depth': 8, 'min_sam..."


## 16. Final Model Performance Comparison

This section compares the baseline classifiers with the Optuna-tuned tree-based models using cross-validation and 2023–2025 holdout test performance.

In [24]:
# ==================================================
# Step 16: Final Model Performance Comparison
# ==================================================

# --------------------------------------------------
# Combine baseline cross-validation results
# --------------------------------------------------

cv_results_list = [
    naive_baseline_cv_results,
    logistic_regression_cv_results,
    decision_tree_cv_results,
    random_forest_cv_results
]

if xgboost_cv_results is not None:
    cv_results_list.append(xgboost_cv_results)


# --------------------------------------------------
# Add Optuna-tuned model cross-validation results
# --------------------------------------------------

if "optuna_model_outputs" in globals():
    for model_name, model_output in optuna_model_outputs.items():
        cv_results_list.append(model_output["cv_results"])


all_cv_results = pd.concat(
    cv_results_list,
    ignore_index=True
)


# --------------------------------------------------
# Summarize cross-validation results
# --------------------------------------------------

cv_summary = (
    all_cv_results
    .groupby("model")
    .agg(
        cv_accuracy_mean=("accuracy", "mean"),
        cv_accuracy_std=("accuracy", "std"),
        cv_macro_f1_mean=("macro_f1", "mean"),
        cv_macro_f1_std=("macro_f1", "std"),
        cv_weighted_f1_mean=("weighted_f1", "mean"),
        cv_weighted_f1_std=("weighted_f1", "std")
    )
    .reset_index()
)


# --------------------------------------------------
# Combine baseline holdout test results
# --------------------------------------------------

test_metrics_list = [
    naive_baseline_test_metrics,
    logistic_regression_test_metrics,
    decision_tree_test_metrics,
    random_forest_test_metrics
]

if xgboost_test_metrics is not None:
    test_metrics_list.append(xgboost_test_metrics)


# --------------------------------------------------
# Add Optuna-tuned model holdout test results
# --------------------------------------------------

if "optuna_model_outputs" in globals():
    for model_name, model_output in optuna_model_outputs.items():
        test_metrics_list.append(model_output["test_metrics"])


test_summary = pd.DataFrame(test_metrics_list)

test_summary = test_summary.rename(columns={
    "accuracy": "test_accuracy",
    "macro_precision": "test_macro_precision",
    "macro_recall": "test_macro_recall",
    "macro_f1": "test_macro_f1",
    "weighted_f1": "test_weighted_f1"
})


# --------------------------------------------------
# Merge CV and holdout test results
# --------------------------------------------------

model_comparison = cv_summary.merge(
    test_summary,
    on="model",
    how="left"
)


# --------------------------------------------------
# Reorder columns
# --------------------------------------------------

model_comparison = model_comparison[
    [
        "model",
        "cv_accuracy_mean",
        "cv_accuracy_std",
        "cv_macro_f1_mean",
        "cv_macro_f1_std",
        "cv_weighted_f1_mean",
        "cv_weighted_f1_std",
        "test_accuracy",
        "test_macro_precision",
        "test_macro_recall",
        "test_macro_f1",
        "test_weighted_f1"
    ]
]


# --------------------------------------------------
# Round results for readability
# --------------------------------------------------

numeric_columns = model_comparison.select_dtypes(include=["float", "int"]).columns

model_comparison[numeric_columns] = model_comparison[numeric_columns].round(4)


# --------------------------------------------------
# Sort by holdout macro F1
# --------------------------------------------------

model_comparison = model_comparison.sort_values(
    by="test_macro_f1",
    ascending=False
).reset_index(drop=True)


print("Final model comparison summary:")
display(model_comparison)

Final model comparison summary:


,model,cv_accuracy_mean,cv_accuracy_std,cv_macro_f1_mean,cv_macro_f1_std,cv_weighted_f1_mean,cv_weighted_f1_std,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,test_weighted_f1
0,Logistic Regression,0.7758,0.0922,0.7345,0.1097,0.7639,0.1070,0.7761,0.8038,0.8077,0.7687,0.7622
1,XGBoost (Optuna),0.8117,0.0403,0.7934,0.0877,0.8198,0.0422,0.7313,0.7438,0.7673,0.7218,0.7077
2,Random Forest,0.7669,0.0431,0.7371,0.0568,0.7760,0.0500,0.6418,0.6445,0.6708,0.6321,0.6159
3,XGBoost,0.7938,0.0402,0.7591,0.0725,0.8041,0.0410,0.6468,0.6543,0.6887,0.6221,0.6033
4,Random Forest (Optuna),0.7713,0.0357,0.7469,0.0606,0.7801,0.0433,0.6169,0.6260,0.6448,0.6029,0.5845
5,Decision Tree,0.6397,0.0247,0.5655,0.1091,0.6596,0.0512,0.5473,0.5575,0.5787,0.5295,0.5175
6,Naive Baseline,0.1782,0.1533,0.0932,0.0735,0.0767,0.0893,0.2637,0.0879,0.3333,0.1391,0.1100


### 16.1 Model Comparison Results

The final model comparison shows that Logistic Regression achieved the strongest 2023–2025 holdout performance, with the highest test macro F1 and weighted F1 scores. Although Optuna tuning improved XGBoost compared with its original baseline result, the tuned XGBoost model still did not outperform Logistic Regression on the holdout test set.

This suggests that the simpler Logistic Regression model generalized more reliably to later years. Therefore, Logistic Regression was retained as the final selected classifier for this notebook.

## 17. Best Model Evaluation

This section evaluates the best-performing model using a confusion matrix and class-wise classification report.

In [25]:
# ==================================================
# Step 17: Best Model Evaluation
# ==================================================

# --------------------------------------------------
# Select best model based on holdout test macro F1
# --------------------------------------------------

best_model_name = model_comparison.loc[0, "model"]

print(f"Best model based on test macro F1: {best_model_name}")


# --------------------------------------------------
# Create lookup dictionaries for predictions and fitted models
# --------------------------------------------------

model_predictions_lookup = {
    "Naive Baseline": naive_baseline_test_predictions,
    "Logistic Regression": logistic_regression_test_predictions,
    "Decision Tree": decision_tree_test_predictions,
    "Random Forest": random_forest_test_predictions
}

model_fitted_lookup = {
    "Naive Baseline": naive_baseline_fitted_model,
    "Logistic Regression": logistic_regression_fitted_model,
    "Decision Tree": decision_tree_fitted_model,
    "Random Forest": random_forest_fitted_model
}

if xgboost_test_predictions is not None:
    model_predictions_lookup["XGBoost"] = xgboost_test_predictions
    model_fitted_lookup["XGBoost"] = xgboost_fitted_model

# Add Optuna-tuned models if available
if "optuna_model_outputs" in globals():
    for model_name, model_output in optuna_model_outputs.items():
        model_predictions_lookup[model_name] = model_output["test_predictions"]
        model_fitted_lookup[model_name] = model_output["fitted_model"]


# --------------------------------------------------
# Select predictions and fitted model for best model
# --------------------------------------------------

best_test_predictions = model_predictions_lookup[best_model_name].copy()
best_fitted_model = model_fitted_lookup[best_model_name]


# --------------------------------------------------
# Actual and predicted labels
# --------------------------------------------------

y_test_actual = best_test_predictions["actual_vulnerability_class"]
y_test_predicted = best_test_predictions["predicted_vulnerability_class"]

class_order = ["Low", "Medium", "High"]


# --------------------------------------------------
# Confusion matrix
# --------------------------------------------------

best_confusion_matrix = confusion_matrix(
    y_test_actual,
    y_test_predicted,
    labels=class_order
)

best_confusion_matrix_df = pd.DataFrame(
    best_confusion_matrix,
    index=[f"Actual {label}" for label in class_order],
    columns=[f"Predicted {label}" for label in class_order]
)

print("\nConfusion matrix:")
display(best_confusion_matrix_df)


# --------------------------------------------------
# Classification report
# --------------------------------------------------

best_classification_report = classification_report(
    y_test_actual,
    y_test_predicted,
    labels=class_order,
    output_dict=True,
    zero_division=0
)

best_classification_report_df = pd.DataFrame(best_classification_report).T

print("\nClassification report:")
display(best_classification_report_df.round(4))

Best model based on test macro F1: Logistic Regression

Confusion matrix:


,Predicted Low,Predicted Medium,Predicted High
Actual Low,53,0,0
Actual Medium,33,38,8
Actual High,0,4,65



Classification report:


,precision,recall,f1-score,support
Low,0.6163,1.0000,0.7626,53.0000
Medium,0.9048,0.4810,0.6281,79.0000
High,0.8904,0.9420,0.9155,69.0000
accuracy,0.7761,0.7761,0.7761,0.7761
macro avg,0.8038,0.8077,0.7687,201.0000
weighted avg,0.8238,0.7761,0.7622,201.0000


### 17.1 Best Model Evaluation Results

Logistic Regression was selected as the best-performing model based on holdout test macro F1. The model achieved a test accuracy of 0.7761 and a macro F1 score of 0.7687 on the 2023–2025 holdout period.

The confusion matrix shows that the model clearly separated Low and High vulnerability observations. No Low observations were predicted as High, and no High observations were predicted as Low. Most classification difficulty occurred in the Medium class, which is expected because Medium vulnerability represents a transition category between Low and High vulnerability.

Overall, the results suggest that the final model can reliably identify the strongest Low and High vulnerability patterns while showing more uncertainty for borderline Medium cases.

## 18. Logistic Regression Feature Coefficients

This section extracts standardized feature coefficients from the final Logistic Regression model.

In [26]:
# ==================================================
# Step 18: Logistic Regression Feature Coefficients
# ==================================================

# --------------------------------------------------
# Confirm final model is Logistic Regression
# --------------------------------------------------

if best_model_name != "Logistic Regression":
    print(f"Best model is {best_model_name}, so Logistic Regression coefficients are not extracted as the final model.")
else:
    print("Extracting coefficients from the final Logistic Regression model.")


# --------------------------------------------------
# Extract fitted Logistic Regression steps
# --------------------------------------------------

logistic_imputer = logistic_regression_fitted_model.named_steps["imputer"]
logistic_scaler = logistic_regression_fitted_model.named_steps["scaler"]
logistic_classifier = logistic_regression_fitted_model.named_steps["classifier"]


# --------------------------------------------------
# Create coefficient table
# --------------------------------------------------

model_classes = logistic_classifier.classes_

coefficient_rows = []

for class_index, class_label in enumerate(model_classes):
    
    class_coefficients = logistic_classifier.coef_[class_index]
    
    for feature_name, coefficient_value in zip(numeric_feature_columns, class_coefficients):
        
        if coefficient_value > 0:
            coefficient_direction = "Positive"
        elif coefficient_value < 0:
            coefficient_direction = "Negative"
        else:
            coefficient_direction = "Zero"
        
        coefficient_rows.append({
            "class": class_label,
            "feature": feature_name,
            "coefficient": coefficient_value,
            "absolute_coefficient": abs(coefficient_value),
            "direction": coefficient_direction
        })

logistic_coefficient_table = pd.DataFrame(coefficient_rows)

logistic_coefficient_table["coefficient"] = logistic_coefficient_table["coefficient"].round(4)
logistic_coefficient_table["absolute_coefficient"] = logistic_coefficient_table["absolute_coefficient"].round(4)


# --------------------------------------------------
# Top features by absolute coefficient for each class
# --------------------------------------------------

top_logistic_coefficients = (
    logistic_coefficient_table
    .sort_values(["class", "absolute_coefficient"], ascending=[True, False])
    .groupby("class")
    .head(15)
    .reset_index(drop=True)
)

print("Top Logistic Regression coefficients by class:")
display(top_logistic_coefficients)

Extracting coefficients from the final Logistic Regression model.
Top Logistic Regression coefficients by class:


,class,feature,coefficient,absolute_coefficient,direction
0,High,RESL_SCORE,-2.9307,2.9307,Negative
1,High,CFLD_RISKS,2.5871,2.5871,Positive
2,High,neighbor_high_growth_share,1.8181,1.8181,Positive
3,High,nfip_avg_claim_payment,1.3848,1.3848,Positive
4,High,nfip_claim_year_indicator,1.1128,1.1128,Positive
5,High,neighbor_high_volatility_share,1.1041,1.1041,Positive
6,High,HRCN_RISKS,1.0960,1.0960,Positive
7,High,SOVI_SCORE,0.9710,0.9710,Positive
8,High,price_to_income_ratio,0.9082,0.9082,Positive
9,High,hurricane_disaster_count,0.8106,0.8106,Positive


In [27]:
# ==================================================
# Step 18.1: Top High Vulnerability Coefficients
# ==================================================

high_vulnerability_coefficients = (
    logistic_coefficient_table[
        logistic_coefficient_table["class"] == "High"
    ]
    .copy()
)

top_positive_high_coefficients = (
    high_vulnerability_coefficients
    .sort_values("coefficient", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

top_negative_high_coefficients = (
    high_vulnerability_coefficients
    .sort_values("coefficient", ascending=True)
    .head(15)
    .reset_index(drop=True)
)

print("Top positive coefficients for High vulnerability:")
display(top_positive_high_coefficients)

print("Top negative coefficients for High vulnerability:")
display(top_negative_high_coefficients)

Top positive coefficients for High vulnerability:


,class,feature,coefficient,absolute_coefficient,direction
0,High,CFLD_RISKS,2.5871,2.5871,Positive
1,High,neighbor_high_growth_share,1.8181,1.8181,Positive
2,High,nfip_avg_claim_payment,1.3848,1.3848,Positive
3,High,nfip_claim_year_indicator,1.1128,1.1128,Positive
4,High,neighbor_high_volatility_share,1.1041,1.1041,Positive
5,High,HRCN_RISKS,1.0960,1.0960,Positive
6,High,SOVI_SCORE,0.9710,0.9710,Positive
7,High,price_to_income_ratio,0.9082,0.9082,Positive
8,High,hurricane_disaster_count,0.8106,0.8106,Positive
9,High,neighbor_avg_price_growth_pct,0.7850,0.7850,Positive


Top negative coefficients for High vulnerability:


,class,feature,coefficient,absolute_coefficient,direction
0,High,RESL_SCORE,-2.9307,2.9307,Negative
1,High,SizeRank,-0.7768,0.7768,Negative
2,High,median_household_income,-0.6445,0.6445,Negative
3,High,Year,-0.5564,0.5564,Negative
4,High,acs_median_home_value,-0.4000,0.4000,Negative
5,High,nfip_total_icc_payment,-0.3626,0.3626,Negative
6,High,severe_storm_disaster_count,-0.3066,0.3066,Negative
7,High,owner_occupied_share,-0.2911,0.2911,Negative
8,High,neighbor_count,-0.1862,0.1862,Negative
9,High,baseline_housing_price,-0.1807,0.1807,Negative


### 18.2 Coefficient Results

The coefficient results show that High vulnerability is mainly linked with higher flood risk, hurricane risk, NFIP claim burden, social vulnerability, affordability stress, disaster history, and neighboring housing pressure.

Higher resilience and household income reduce the likelihood of High vulnerability. The Medium class has weaker coefficients overall, which supports its role as a transition category between Low and High vulnerability.

## 18. Tree-Based Feature Importance

This section extracts feature importance from the tree-based models.

In [28]:
# ==================================================
# Step 18: Tree-Based Feature Importance
# ==================================================

feature_importance_tables = []

# --------------------------------------------------
# Decision Tree feature importance
# --------------------------------------------------

decision_tree_classifier = decision_tree_fitted_model.named_steps["classifier"]

decision_tree_importance = pd.DataFrame({
    "model": "Decision Tree",
    "feature": numeric_feature_columns,
    "importance": decision_tree_classifier.feature_importances_
})

feature_importance_tables.append(decision_tree_importance)

# --------------------------------------------------
# Random Forest feature importance
# --------------------------------------------------

random_forest_classifier = random_forest_fitted_model.named_steps["classifier"]

random_forest_importance = pd.DataFrame({
    "model": "Random Forest",
    "feature": numeric_feature_columns,
    "importance": random_forest_classifier.feature_importances_
})

feature_importance_tables.append(random_forest_importance)

# --------------------------------------------------
# XGBoost feature importance
# --------------------------------------------------

if xgboost_fitted_model is not None:
    xgboost_classifier = xgboost_fitted_model.named_steps["classifier"]
    
    xgboost_importance = pd.DataFrame({
        "model": "XGBoost",
        "feature": numeric_feature_columns,
        "importance": xgboost_classifier.feature_importances_
    })
    
    feature_importance_tables.append(xgboost_importance)

# --------------------------------------------------
# Combine importance tables
# --------------------------------------------------

tree_feature_importance = pd.concat(
    feature_importance_tables,
    ignore_index=True
)

tree_feature_importance = tree_feature_importance.sort_values(
    ["model", "importance"],
    ascending=[True, False]
).reset_index(drop=True)

# Top 15 features from each tree-based model
top_tree_feature_importance = (
    tree_feature_importance
    .groupby("model")
    .head(15)
    .reset_index(drop=True)
)

print("Top tree-based feature importance values:")
display(top_tree_feature_importance)

Top tree-based feature importance values:


,model,feature,importance
0,Decision Tree,CFLD_RISKS,0.2831
1,Decision Tree,neighbor_avg_price_growth_pct,0.2805
2,Decision Tree,RESL_SCORE,0.0999
3,Decision Tree,climate_disaster_count,0.0615
4,Decision Tree,annual_price_growth_dollar,0.0444
5,Decision Tree,nfip_recent_3yr_claim_payment,0.0418
6,Decision Tree,SOVI_SCORE,0.0411
7,Decision Tree,neighbor_avg_housing_price,0.0383
8,Decision Tree,nfip_cumulative_claim_payment,0.0374
9,Decision Tree,nfip_avg_claim_payment,0.0251


Tree-based feature importance results show that climate exposure, resilience, social vulnerability, NFIP insurance-loss history, housing market pressure, and spatial spillover variables were repeatedly selected as important predictors. Although tree-based models did not outperform Logistic Regression on the holdout test set, their feature importance results support the broader vulnerability framework by showing that classification decisions are influenced by multiple dimensions of climate-housing vulnerability.

## 19. Save Model Outputs

This section saves the model comparison table, test predictions, and feature importance outputs.

In [29]:
# ==================================================
# Step 19: Save Model Outputs
# ==================================================

# --------------------------------------------------
# Output file paths
# --------------------------------------------------

model_comparison_path = (
    RESULTS_TABLES
    / "vulnerability_classification_model_comparison.csv"
)

best_predictions_path = (
    RESULTS_TABLES
    / "best_model_vulnerability_classification_predictions.csv"
)

logistic_coefficients_path = (
    RESULTS_TABLES
    / "logistic_regression_feature_coefficients.csv"
)

tree_feature_importance_path = (
    RESULTS_TABLES
    / "tree_based_feature_importance.csv"
)

# --------------------------------------------------
# Save outputs
# --------------------------------------------------

model_comparison.to_csv(model_comparison_path, index=False)
best_test_predictions.to_csv(best_predictions_path, index=False)
logistic_coefficient_table.to_csv(logistic_coefficients_path, index=False)
tree_feature_importance.to_csv(tree_feature_importance_path, index=False)

print("Model outputs saved successfully.")
print(f"Model comparison: {model_comparison_path}")
print(f"Best model predictions: {best_predictions_path}")
print(f"Logistic coefficients: {logistic_coefficients_path}")
print(f"Tree feature importance: {tree_feature_importance_path}")

Model outputs saved successfully.
Model comparison: ..\results\tables\vulnerability_classification_model_comparison.csv
Best model predictions: ..\results\tables\best_model_vulnerability_classification_predictions.csv
Logistic coefficients: ..\results\tables\logistic_regression_feature_coefficients.csv
Tree feature importance: ..\results\tables\tree_based_feature_importance.csv


## 20. Export Final Logistic Regression Artifacts for Explainability

This final export step preserves the exact fitted Logistic Regression pipeline and the information required to reproduce its 2023–2025 holdout predictions in the explainability notebook. It saves the ordered feature list, class order, temporal split settings, cross-validation fold definitions, and class probabilities for every test county-year. The existing model-comparison and interpretation tables are also saved again so that all outputs reflect the same clean top-to-bottom notebook run.


In [30]:
# ==================================================
# Step 20: Export Final Logistic Regression Artifacts
# ==================================================

import json
from datetime import datetime, timezone

import joblib
import sklearn

# --------------------------------------------------
# Validate required fitted objects
# --------------------------------------------------

required_objects = [
    "logistic_regression_fitted_model",
    "numeric_feature_columns",
    "county_year_data_sorted",
    "test_mask",
    "X_test",
    "y_test",
    "model_comparison",
    "best_test_predictions",
    "logistic_coefficient_table",
    "tree_feature_importance"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the notebook from top to bottom before exporting. "
        f"Missing objects: {missing_objects}"
    )

# --------------------------------------------------
# Recover fitted model information
# --------------------------------------------------

final_logistic_pipeline = logistic_regression_fitted_model
final_logistic_classifier = final_logistic_pipeline.named_steps["classifier"]
logistic_class_order = [str(label) for label in final_logistic_classifier.classes_]

# Generate predictions and probabilities from the exact fitted pipeline
logistic_test_predictions = final_logistic_pipeline.predict(X_test)
logistic_test_probabilities = final_logistic_pipeline.predict_proba(X_test)

# --------------------------------------------------
# Build test prediction table with identifiers
# --------------------------------------------------

logistic_probability_table = (
    county_year_data_sorted
    .loc[test_mask, ["Year", "STCOFIPS", "RegionName"]]
    .reset_index(drop=True)
)

logistic_probability_table["actual_vulnerability_class"] = (
    y_test.reset_index(drop=True).astype(str)
)

logistic_probability_table["predicted_vulnerability_class"] = (
    pd.Series(logistic_test_predictions).astype(str)
)

for class_index, class_label in enumerate(logistic_class_order):
    probability_column = f"probability_{class_label.lower()}"
    logistic_probability_table[probability_column] = (
        logistic_test_probabilities[:, class_index]
    )

logistic_probability_table["predicted_probability"] = (
    logistic_test_probabilities.max(axis=1)
)

sorted_probabilities = np.sort(logistic_test_probabilities, axis=1)
logistic_probability_table["probability_margin"] = (
    sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
)

logistic_probability_table["prediction_correct"] = (
    logistic_probability_table["actual_vulnerability_class"]
    == logistic_probability_table["predicted_vulnerability_class"]
)

# --------------------------------------------------
# Define output paths
# --------------------------------------------------

logistic_pipeline_path = (
    RESULTS_MODELS
    / "final_logistic_regression_vulnerability_pipeline.joblib"
)

logistic_feature_list_path = (
    RESULTS_MODELS
    / "final_logistic_regression_feature_list.json"
)

logistic_metadata_path = (
    RESULTS_MODELS
    / "final_logistic_regression_metadata.json"
)

logistic_probability_path = (
    RESULTS_TABLES
    / "logistic_regression_test_predictions_with_probabilities.csv"
)

# --------------------------------------------------
# Save exact fitted pipeline
# --------------------------------------------------

joblib.dump(
    final_logistic_pipeline,
    logistic_pipeline_path
)

# --------------------------------------------------
# Save ordered feature list
# --------------------------------------------------

feature_list_export = {
    "feature_count": int(len(numeric_feature_columns)),
    "features": [str(feature) for feature in numeric_feature_columns]
}

with logistic_feature_list_path.open("w", encoding="utf-8") as file:
    json.dump(feature_list_export, file, indent=2)

# --------------------------------------------------
# Save modelling and split metadata
# --------------------------------------------------

metadata_export = {
    "model_name": "Logistic Regression",
    "target_column": str(target_column),
    "class_order": logistic_class_order,
    "feature_count": int(len(numeric_feature_columns)),
    "training_year_start": int(train_year_range[0]),
    "training_year_end": int(train_year_range[1]),
    "testing_year_start": int(test_year_range[0]),
    "testing_year_end": int(test_year_range[1]),
    "train_end_year_setting": int(TRAIN_END_YEAR),
    "test_start_year_setting": int(TEST_START_YEAR),
    "random_state": int(RANDOM_STATE),
    "training_observations": int(len(X_train)),
    "testing_observations": int(len(X_test)),
    "cross_validation_folds": cv_year_folds,
    "scikit_learn_version": sklearn.__version__,
    "exported_at_utc": datetime.now(timezone.utc).isoformat()
}

with logistic_metadata_path.open("w", encoding="utf-8") as file:
    json.dump(metadata_export, file, indent=2)

# --------------------------------------------------
# Save test probabilities
# --------------------------------------------------

logistic_probability_table.to_csv(
    logistic_probability_path,
    index=False
)

# --------------------------------------------------
# Re-save existing final outputs from this clean run
# --------------------------------------------------

model_comparison.to_csv(model_comparison_path, index=False)
best_test_predictions.to_csv(best_predictions_path, index=False)
logistic_coefficient_table.to_csv(logistic_coefficients_path, index=False)
tree_feature_importance.to_csv(tree_feature_importance_path, index=False)

# --------------------------------------------------
# Verify exported artifacts
# --------------------------------------------------

exported_artifacts = pd.DataFrame({
    "artifact": [
        "Fitted Logistic Regression pipeline",
        "Ordered Logistic Regression feature list",
        "Logistic Regression metadata",
        "Test predictions with class probabilities",
        "Model comparison table",
        "Best-model prediction table",
        "Logistic Regression coefficient table",
        "Tree-based feature-importance table"
    ],
    "path": [
        str(logistic_pipeline_path),
        str(logistic_feature_list_path),
        str(logistic_metadata_path),
        str(logistic_probability_path),
        str(model_comparison_path),
        str(best_predictions_path),
        str(logistic_coefficients_path),
        str(tree_feature_importance_path)
    ]
})

print("Final Logistic Regression explainability artifacts exported successfully.")
print(f"Saved fitted pipeline: {logistic_pipeline_path}")
print(f"Saved ordered feature list: {logistic_feature_list_path}")
print(f"Saved model metadata: {logistic_metadata_path}")
print(f"Saved test probabilities: {logistic_probability_path}")
print(f"Probability-table shape: {logistic_probability_table.shape}")
print(f"Class order: {logistic_class_order}")

display(exported_artifacts)
display(logistic_probability_table.head())


Final Logistic Regression explainability artifacts exported successfully.
Saved fitted pipeline: ..\results\models\final_logistic_regression_vulnerability_pipeline.joblib
Saved ordered feature list: ..\results\models\final_logistic_regression_feature_list.json
Saved model metadata: ..\results\models\final_logistic_regression_metadata.json
Saved test probabilities: ..\results\tables\logistic_regression_test_predictions_with_probabilities.csv
Probability-table shape: (201, 11)
Class order: ['High', 'Low', 'Medium']


,artifact,path
0,Fitted Logistic Regression pipeline,..\results\models\final_logistic_regression_vu...
1,Ordered Logistic Regression feature list,..\results\models\final_logistic_regression_fe...
2,Logistic Regression metadata,..\results\models\final_logistic_regression_me...
3,Test predictions with class probabilities,..\results\tables\logistic_regression_test_pre...
4,Model comparison table,..\results\tables\vulnerability_classification...
5,Best-model prediction table,..\results\tables\best_model_vulnerability_cla...
6,Logistic Regression coefficient table,..\results\tables\logistic_regression_feature_...
7,Tree-based feature-importance table,..\results\tables\tree_based_feature_importanc...


,Year,STCOFIPS,RegionName,actual_vulnerability_class,predicted_vulnerability_class,probability_high,probability_low,probability_medium,predicted_probability,probability_margin,prediction_correct
0,2023,12001,Alachua County,Low,Low,0.0000,0.9906,0.0094,0.9906,0.9813,True
1,2023,12003,Baker County,Low,Low,0.0000,1.0000,0.0000,1.0000,1.0000,True
2,2023,12005,Bay County,High,High,0.8354,0.0008,0.1638,0.8354,0.6716,True
3,2023,12007,Bradford County,Low,Low,0.0000,0.9978,0.0022,0.9978,0.9955,True
4,2023,12009,Brevard County,Medium,Low,0.0007,0.7845,0.2148,0.7845,0.5697,False
